# 03 - Seeds

**You will learn**: what seeds are, `dbt seed`, seed configuration (column types), and how to `ref()` a seed.

**You will build**: the `countries` reference table and `dim_customer`, the first **gold** model.

A **seed** is a small CSV file kept in the dbt project and loaded into the warehouse with `dbt seed`.
Use seeds for **small, rarely changing reference data** (country codes, mappings, category groups) that lives with the code and is reviewed in git.
Do **not** use them for large or frequently changing data: that is what ingestion is for.

In [ ]:
from helpers import *

## 1. Add the seed file

Run the cell to write `seeds/countries.csv`: 31 countries with continent, region, currency and EU membership.
Our customers come from Switzerland, Germany, France, Italy and Austria.

In [ ]:
%%writefile ../../src/seeds/countries.csv
country_code,country_name,continent,region,currency_code,is_eu
CH,Switzerland,Europe,Western Europe,CHF,false
LI,Liechtenstein,Europe,Western Europe,CHF,false
DE,Germany,Europe,Western Europe,EUR,true
FR,France,Europe,Western Europe,EUR,true
AT,Austria,Europe,Western Europe,EUR,true
IT,Italy,Europe,Southern Europe,EUR,true
ES,Spain,Europe,Southern Europe,EUR,true
PT,Portugal,Europe,Southern Europe,EUR,true
NL,Netherlands,Europe,Western Europe,EUR,true
BE,Belgium,Europe,Western Europe,EUR,true
LU,Luxembourg,Europe,Western Europe,EUR,true
IE,Ireland,Europe,Northern Europe,EUR,true
DK,Denmark,Europe,Northern Europe,DKK,true
SE,Sweden,Europe,Northern Europe,SEK,true
FI,Finland,Europe,Northern Europe,EUR,true
NO,Norway,Europe,Northern Europe,NOK,false
GB,United Kingdom,Europe,Northern Europe,GBP,false
PL,Poland,Europe,Eastern Europe,PLN,true
CZ,Czechia,Europe,Eastern Europe,CZK,true
US,United States,North America,Northern America,USD,false
CA,Canada,North America,Northern America,CAD,false
MX,Mexico,North America,Central America,MXN,false
BR,Brazil,South America,South America,BRL,false
AR,Argentina,South America,South America,ARS,false
JP,Japan,Asia,Eastern Asia,JPY,false
CN,China,Asia,Eastern Asia,CNY,false
IN,India,Asia,Southern Asia,INR,false
AE,United Arab Emirates,Asia,Western Asia,AED,false
AU,Australia,Oceania,Australia and New Zealand,AUD,false
NZ,New Zealand,Oceania,Australia and New Zealand,NZD,false
ZA,South Africa,Africa,Southern Africa,ZAR,false


## 2. Configure the column types

dbt guesses the type of every CSV column. `is_eu` would probably become a string, we want a boolean.
Seeds are configured in a YAML file next to the CSV.

**Exercise A.** Complete `seeds/_seeds.yml`: set the column type of `country_code` to `string` and `is_eu` to `boolean`.

In [ ]:
%%writefile ../../src/seeds/_seeds.yml
version: 2

seeds:
  - name: countries
    config:
      column_types:
        # TODO: country_code: string
        # TODO: is_eu: boolean
        currency_code: string


In [ ]:
dbt("seed")

In [ ]:
d = q(f"DESCRIBE silver.{SCHEMA}.countries")
check("is_eu is a boolean", d.set_index("col_name").loc["is_eu", "data_type"] == "boolean",
      "add is_eu: boolean under column_types and re-run dbt('seed')")
d

The seed is now a table `silver.<your_schema>.countries`, exactly like a model.

## 3. Change the seed

**Exercise B.** Add a line for Monaco at the end of `countries.csv`:

```
MC,Monaco,Europe,Western Europe,EUR,false
```

Then run `dbt seed` again and check that the table has 32 rows. dbt reloads the whole file each time.
(If you *change the columns* of a seed, use `dbt seed --full-refresh` to recreate the table.)

In [ ]:
dbt("seed")
q(f"SELECT COUNT(*) AS countries FROM silver.{SCHEMA}.countries")

## 4. Use the seed: `dim_customer`

A seed is used like any model: `{{ ref('countries') }}`. `ref()` also tells dbt that `dim_customer` depends on `countries`
and on `stg_customers`, so they are built first.

**Exercise C.** Build the customer dimension (in the **gold** folder, so it lands in `gold.<your_schema>`).
Left join `stg_customers` to the seed on `country_code` and bring in `country_name`, `continent` and `is_eu`.

In [ ]:
%%writefile ../../src/models/gold/dim_customer.sql
-- Customers enriched with the country attributes from the `countries` seed.
select
    c.customer_id,
    c.full_name,
    c.email,
    c.city,
    c.country_code,
    -- TODO: country_name, continent and is_eu from the seed (alias co)
    c.loyalty_tier,
    c.created_at
from {{ ref('stg_customers') }} as c
-- TODO: left join {{ ref('countries') }} as co on the country code


In [ ]:
dbt("run --select dim_customer")

In [ ]:
check("every customer has a country name", scalar(f"SELECT COUNT(*) FROM gold.{SCHEMA}.dim_customer WHERE country_name IS NULL") == 0,
      "check the join condition")
q(f'''
    SELECT country_name, continent, is_eu, COUNT(*) AS customers
    FROM gold.{SCHEMA}.dim_customer
    GROUP BY 1, 2, 3 ORDER BY customers DESC
''')

In [ ]:
dbt("ls --select +dim_customer --output name")

The lineage of `dim_customer` now shows the source table, the staging model **and** the seed.

## Recap

* Seeds = small CSV reference data, versioned with the code, loaded by `dbt seed`.
* `column_types` fixes the types; `--full-refresh` recreates the table when columns change.
* `ref('seed_name')` works exactly like a model reference.

---

In [ ]:
# restore_checkpoint(3)